In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
from etc.parse_ids import XMLParser

# Get the notebook's directory and go up to project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
data_folder = project_root / "data" / "resources"
PilotDC9 = data_folder / "PilotStudy_Afekta"

graph_gml = data_folder / "generated" / "modified_graph.gml"
human1_xml = data_folder / "Human-GEM.xml"

In [ ]:
G = nx.read_gml(graph_gml)

In [ ]:
chebi_match = pd.read_csv(PilotDC9 / "AFEKTA_chebis.csv")

In [ ]:
name_match = pd.read_excel(PilotDC9 / "Supplementary_Material_2_Results_Dennisse_Avella.xlsx",sheet_name="keyIDs")

In [ ]:
mask = name_match["Idlevel"] <= 2

# Clean empty values or '-' to None
curated = (
    name_match.loc[mask, "CuratedID"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
name_match.loc[mask, "CuratedID"] = curated.where(curated.notna(), None)
# Clean chebi_match
chebi_clean = chebi_match.copy()
chebi_clean["Input name"] = (
    chebi_clean["Input name"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
chebi_clean["CHEBI_ID"] = chebi_clean["CHEBI_ID"].replace({"": None, "-": None})
# Create lookup table from chebi_clean
lookup = (
    chebi_clean.dropna(subset=["Input name"])
    .drop_duplicates(subset=["Input name"], keep="first")
    .set_index("Input name")["CHEBI_ID"]
)
# Map Chebi_IDs to name_match
name_match.loc[mask, "Chebi_ID"] = name_match.loc[mask, "CuratedID"].map(lookup)

In [ ]:
name_match

In [2]:
parser = XMLParser(human1_xml)

In [3]:
df = parser.extract_data()

In [4]:
parser._first_of(df.Identifiers[68], {'kegg.compound'})

In [5]:
df.Identifiers[68]

[['bigg.metabolite', 'nrvnccrn'],
 ['vmhmetabolite', 'nrvnccrn'],
 ['metanetx.chemical', 'MNXM8942'],
 ['inchi',
  'InChI=1S',
  'C31H59NO4',
  'c1-5-6-7-8-9-10-11-12-13-14-15-16-17-18-19-20-21-22-23-24-25-26-31(35)36-29(27-30(33)34)28-32(2,3)4',
  'h12-13,29H,5-11,14-28H2,1-4H3',
  't29-',
  'm0',
  's1']]

In [6]:
df_human1 = parser.get_chebi_numbers()

In [7]:
df_human1 = parser.to_identifier_df()

In [8]:
df_human1

,chebi,kegg,metanetx,vmhmetabolite,hmdb,lipidmaps,pubchem
HUMAN1_ID,,,,,,,
MAM00001c,15389,C00964,MNXM45735,carveol,NaN,NaN,NaN
MAM00001e,15389,C00964,MNXM45735,carveol,NaN,NaN,NaN
MAM00002c,36740,C09880,MNXM163755,appnn,HMDB0006525,NaN,6654
MAM00002e,36740,C09880,MNXM163755,appnn,HMDB0006525,NaN,6654
MAM00003c,78990,NaN,MNXM150165,M00003,NaN,LMFA01030283,NaN
...,...,...,...,...,...,...,...
MAM01316n,60721,NaN,MNXM731626,NaN,NaN,NaN,NaN
MAM00767n,84503,NaN,MNXM733937,NaN,NaN,NaN,NaN
MAM01435m,16755,C02528,MNXM1183,HC00958,HMDB0000518,LMST04010032,10133
